# **CS 584-A: Natural Language Processing**
# **Homework 3**

**Name: Ankit Dhandharia**

**CWID: 20033031**

## **Goals**
The goal of HW3 is for you to get hands-on experience of utilizing Seq2Seq model for machine
translation task. You will get a deeper understanding of how the input sequence is translated to
the output sequence with the Seq2Seq model. The skills of you learnt in this homework will
benefit your understanding of a wide range of NLP tasks beyond machine translation. Please feel
free to use any packages or libraries in your implementation.


## **Task: Machine Translation**

#**1. Data preparation**

In [31]:
!pip install datasets
!pip install transformers
!pip install torchtext
!pip install torch
!pip install tensorflow
!pip install sentencepiece
!pip install sacremoses
!pip install sacrebleu
!pip install ctranslate2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 17.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 12.6 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset
dataset = load_dataset("Muennighoff/flores200", "all")

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [40]:
import random

# Set random seed
random.seed(50)

# Get sentences in both languages
thai_text = dataset['dev']['sentence_tha_Thai']
english_text = dataset['dev']['sentence_eng_Latn']

# Make pairs and select 100 random ones
all_pairs = list(zip(thai_text, english_text))
random_pairs = random.sample(all_pairs, 100)

# Show all pairs
for num, (thai, eng) in enumerate(random_pairs):
    print(f"\nPair {num + 1}")
    print(f"Thai: {thai}")
    print(f"English: {eng}")

Pair 0
Thai: นี่ไม่ใช่กลุ่มอาการขาดทักษะด้านการเรียนรู้ แต่เป็นความผิดปกติทางการเรียนรู้ โรคนี้ "ส่งผลกระทบต่อเด็กร้อยละ 3 ถึง 5 ซึ่งอาจเป็นเด็กอเมริกันมากถึง 2 ล้านคน"
English: It is not a learning disability, it is a learning disorder; it "affects 3 to 5 percent of all children, perhaps as many as 2 million American children".

Pair 1
Thai: ผู้ที่ไม่คุ้นเคยกับศัพท์เฉพาะทางการแพทย์คิดว่า คำว่าติดเชื้อและโรคติดต่อนั้นมีความหมายต่างกัน
English: For those unfamiliar with medical jargon, the words infectious and contagious have distinct meanings.

Pair 2
Thai: เขาตั้งข้อสังเกตว่า งานวิจัยบางชิ้นชี้ให้เห็นว่าโรคนี้จำเป็นต้องมีอันตรายน้อยลงก่อนที่มันจะแพร่ระบาดไปทั่วโลก
English: Some studies suggest that the disease must become less lethal before it can cause a global epidemic, he noted.

Pair 3
Thai: นักวิทยาศาสตร์รายอื่น ๆ แย้งว่าพืชเหล่านี้มีพิษร้ายแรง จึงไม่น่าเป็นไปได้ที่จะมีไดโนเสาร์ตัวใดกินพวกมันได้ แม้ว่าในปัจจุบันสลอธและสัตว์อื่น ๆ เช่น นกแก้ว (ลูกหลานของไดโนเสาร์) จะสามารถกินใบไม้

# **2. Machine Translation with Seq2Seq model (65 points, 20 points for each model, 5 points for data statistics)**

## **2.1 OPUS-MT**

In [41]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Get the Thai-to-English translation model
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-th-en")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-th-en")

# Simple translation function
def translate_thai(thai_text):
    tokens = tokenizer(thai_text, return_tensors="pt")
    result = model.generate(**tokens)
    return tokenizer.batch_decode(result, skip_special_tokens=True)[0]

# Go through each pair and translate
for i, (thai, eng) in enumerate(random_pairs):
    thai_translated = translate_thai(thai)
    print(f"\nSentence {i + 1}")
    print(f"Thai: {thai}")
    print(f"English (Original): {eng}")
    print(f"English (Translated): {thai_translated}")


Sentence 1:
Thai:       นี่ไม่ใช่กลุ่มอาการขาดทักษะด้านการเรียนรู้ แต่เป็นความผิดปกติทางการเรียนรู้ โรคนี้ "ส่งผลกระทบต่อเด็กร้อยละ 3 ถึง 5 ซึ่งอาจเป็นเด็กอเมริกันมากถึง 2 ล้านคน"
English:    It is not a learning disability, it is a learning disorder; it "affects 3 to 5 percent of all children, perhaps as many as 2 million American children".
Translated: This isn't a group of learning disability, it's a learning disorder, it's an effect on children from three to five, which is probably two million American children.

Sentence 2:
Thai:       ผู้ที่ไม่คุ้นเคยกับศัพท์เฉพาะทางการแพทย์คิดว่า คำว่าติดเชื้อและโรคติดต่อนั้นมีความหมายต่างกัน
English:    For those unfamiliar with medical jargon, the words infectious and contagious have distinct meanings.
Translated: Those who aren't familiar with medical terms think that infection and infectious diseases have different meanings.

Sentence 3:
Thai:       เขาตั้งข้อสังเกตว่า งานวิจัยบางชิ้นชี้ให้เห็นว่าโรคนี้จำเป็นต้องมีอันตรายน้อยลงก่อนที่มันจะแ

## **2.2 M2M-100**

In [42]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# Get the M2M100 model and tokenizer
model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# Set Thai as source language
tokenizer.src_lang = "th"

# Simple translation function
def translate_m2m(thai_text):
    tokens = tokenizer(thai_text, return_tensors="pt")
    generated = model.generate(**tokens, forced_bos_token_id=tokenizer.get_lang_id("en"))
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

# Translate each pair
for i, (thai, eng) in enumerate(random_pairs):
    m2m_translation = translate_m2m(thai)
    print(f"\nPair {i + 1}:")
    print(f"Thai:     {thai}")
    print(f"English:  {eng}")
    print(f"M2M:      {m2m_translation}")

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]


Pair 1:
Thai:       นี่ไม่ใช่กลุ่มอาการขาดทักษะด้านการเรียนรู้ แต่เป็นความผิดปกติทางการเรียนรู้ โรคนี้ "ส่งผลกระทบต่อเด็กร้อยละ 3 ถึง 5 ซึ่งอาจเป็นเด็กอเมริกันมากถึง 2 ล้านคน"
English:    It is not a learning disability, it is a learning disorder; it "affects 3 to 5 percent of all children, perhaps as many as 2 million American children".
M2M:        This is not a group of learning failure, but a learning disorder. This disease "affects 3 to 5 per child, which may be up to 2 million American children."

Pair 2:
Thai:       ผู้ที่ไม่คุ้นเคยกับศัพท์เฉพาะทางการแพทย์คิดว่า คำว่าติดเชื้อและโรคติดต่อนั้นมีความหมายต่างกัน
English:    For those unfamiliar with medical jargon, the words infectious and contagious have distinct meanings.
M2M:        People who are not familiar with the specific medical term think that the word infection and contact disease have different meanings.

Pair 3:
Thai:       เขาตั้งข้อสังเกตว่า งานวิจัยบางชิ้นชี้ให้เห็นว่าโรคนี้จำเป็นต้องมีอันตรายน้อยลงก่อนที่มันจะแพร่

## **2.3 MBART-50**

In [43]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

# Get the mBART model and tokenizer
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

# Set Thai as source language
tokenizer.src_lang = "th_TH"

# Simple translation function
def translate_mbart(thai_text):
    tokens = tokenizer(thai_text, return_tensors="pt")
    generated = model.generate(**tokens, forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"])
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

# Translate each pair
for i, (thai, eng) in enumerate(random_pairs):
    mbart_translation = translate_mbart(thai)
    print(f"\nPair {i + 1}:")
    print(f"Thai:     {thai}")
    print(f"English:  {eng}")
    print(f"mBART:    {mbart_translation}")


Pair 1:
Thai:       นี่ไม่ใช่กลุ่มอาการขาดทักษะด้านการเรียนรู้ แต่เป็นความผิดปกติทางการเรียนรู้ โรคนี้ "ส่งผลกระทบต่อเด็กร้อยละ 3 ถึง 5 ซึ่งอาจเป็นเด็กอเมริกันมากถึง 2 ล้านคน"
English:    It is not a learning disability, it is a learning disorder; it "affects 3 to 5 percent of all children, perhaps as many as 2 million American children".
mBART:      We're talking about skills shortages, diseases, diseases that affect three to five percent of Americans, maybe two to two percent of Americans.

Pair 2:
Thai:       ผู้ที่ไม่คุ้นเคยกับศัพท์เฉพาะทางการแพทย์คิดว่า คำว่าติดเชื้อและโรคติดต่อนั้นมีความหมายต่างกัน
English:    For those unfamiliar with medical jargon, the words infectious and contagious have distinct meanings.
mBART:      It used to be synonymous with specialized medicines, and diseases were synonymous.

Pair 3:
Thai:       เขาตั้งข้อสังเกตว่า งานวิจัยบางชิ้นชี้ให้เห็นว่าโรคนี้จำเป็นต้องมีอันตรายน้อยลงก่อนที่มันจะแพร่ระบาดไปทั่วโลก
English:    Some studies suggest that the disea

## **2.4 Data statistics**

In [45]:
import pandas as pd

# Simple function to get text length statistics
def get_text_stats(text_list):
    lengths = [len(text) for text in text_list]
    return {
        'min': min(lengths),
        'max': max(lengths),
        'avg': sum(lengths) // len(lengths)
    }

# Create table for source and target text
source_target_table = pd.DataFrame({
    'Sentences': ['Thai', 'English'],
    'Min length': [get_text_stats(thai_text)['min'], get_text_stats(english_text)['min']],
    'Avg length': [get_text_stats(thai_text)['avg'], get_text_stats(english_text)['avg']],
    'Max length': [get_text_stats(thai_text)['max'], get_text_stats(english_text)['max']]
})

# Create table for model translations
models_table = pd.DataFrame({
    'Model': ['OPUS-MT', 'M2M-100', 'MBART-50'],
    'Min length': [get_text_stats(opus_translations)['min'],
                  get_text_stats(m2m_translations)['min'],
                  get_text_stats(mbart_translations)['min']],
    'Avg length': [get_text_stats(opus_translations)['avg'],
                  get_text_stats(m2m_translations)['avg'],
                  get_text_stats(mbart_translations)['avg']],
    'Max length': [get_text_stats(opus_translations)['max'],
                  get_text_stats(m2m_translations)['max'],
                  get_text_stats(mbart_translations)['max']]
})

In [46]:
source_target_table

,Sentences,Min length,Avg length,Max length
0,Source,33,126,287
1,Target,28,132,273


In [47]:
models_table

,Model,Min length,Avg length,Max length
0,OPUS-MT,28,128,267
1,M2M-100,28,143,311
2,MBART-50,27,115,239


# **3. Results analysis and evaluation**

## **3.1 Results of each model**

In [48]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.translate import bleu_score

# Get NLTK data
nltk.download('punkt_tab')

# Process reference (original English) translations
english_tokens = [[word_tokenize(sent)] for sent in english_text]

# Process and score each model's translations
translation_sets = {
    'OPUS-MT': opus_translations,
    'M2M-100': m2m_translations,
    'MBART-50': mbart_translations
}

# Calculate and show BLEU scores
for model_name, translated_text in translation_sets.items():
    # Tokenize translated sentences
    model_tokens = [word_tokenize(sent) for sent in translated_text]

    # Get BLEU score
    bleu = bleu_score.corpus_bleu(english_tokens, model_tokens)

    # Print result
    print(f"\nBLEU score for {model_name}: {bleu:.4f}")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



BLEU score for OPUS-MT: 0.2516

BLEU score for M2M-100: 0.2442

BLEU score for MBART-50: 0.1319


## **3.2 Results discussion**

The OPUS-MT model performed best with a BLEU score of 0.2516, showing its strength in Thai-English translation. M2M-100 came very close with a score of 0.2442, which is impressive for a multilingual model. MBART-50 scored much lower at 0.1319, possibly because its broader language coverage affected its Thai-English translation quality.
